# C12-classical-models — Practice p12 — Solution


The forest is fitted on training rows only. Its required prediction is reconstructed from the 25 member labels, with the smaller class on a tie.


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier

rng_p12 = np.random.default_rng(20260804)
X_train_p12 = rng_p12.normal(size=(80, 5)).astype(np.float64)
y_train_p12 = ((X_train_p12[:, 0] * X_train_p12[:, 1] +
                0.5 * X_train_p12[:, 2]) > 0).astype(np.int64)
X_probe_p12 = np.array([[0.,0.,0.,0.,0.],[1.,1.,0.,0.,0.],
                        [-1.,1.,1.,0.,0.]], dtype=np.float64)


def fit_seeded_forest(X_train, y_train, X_probe):
    if not all(isinstance(a, np.ndarray) for a in (X_train, y_train, X_probe)):
        raise ValueError("inputs must be arrays")
    if X_train.dtype != np.float64 or X_probe.dtype != np.float64 or not np.issubdtype(y_train.dtype, np.integer):
        raise ValueError("invalid dtypes")
    if X_train.ndim != 2 or X_probe.ndim != 2 or y_train.ndim != 1 or y_train.shape != (X_train.shape[0],):
        raise ValueError("invalid shapes")
    if X_train.shape[0] < 2 or X_train.shape[1] < 1 or X_probe.shape[1] != X_train.shape[1]:
        raise ValueError("invalid dimensions")
    if not np.isfinite(X_train).all() or not np.isfinite(X_probe).all() or set(np.unique(y_train)) != {0, 1}:
        raise ValueError("invalid values")
    model = RandomForestClassifier(n_estimators=25, max_depth=4, max_features="sqrt",
                                   bootstrap=True, random_state=20260804)
    model.fit(X_train, y_train)
    members = np.vstack([tree.predict(X_probe) for tree in model.estimators_]).astype(np.int64)
    count_one = members.sum(axis=0)
    predictions = (count_one > members.shape[0] / 2).astype(np.int64)
    return {"model": model, "predictions": predictions,
            "probabilities": model.predict_proba(X_probe),
            "member_predictions": members,
            "feature_importances": model.feature_importances_.astype(np.float64, copy=True)}


result_p12 = fit_seeded_forest(X_train_p12, y_train_p12, X_probe_p12)


### Answer check


In [ ]:
ATOL = 1e-12
RTOL = 1e-10
assert np.array_equal(result_p12["predictions"], [0, 1, 1])
assert result_p12["member_predictions"].shape == (25, 3)
assert np.unique(result_p12["member_predictions"], axis=0).shape[0] >= 2
assert np.allclose(result_p12["probabilities"].sum(axis=1), 1.0, atol=ATOL, rtol=RTOL)
assert np.isclose(result_p12["feature_importances"].sum(), 1.0, atol=ATOL, rtol=RTOL)
repeat_p12 = fit_seeded_forest(X_train_p12, y_train_p12, X_probe_p12)
assert np.array_equal(result_p12["member_predictions"], repeat_p12["member_predictions"])
assert np.allclose(result_p12["probabilities"], repeat_p12["probabilities"], atol=ATOL, rtol=RTOL)
